# Notebook 2 — Lead Optimization Agent Loop

**This is the main notebook.** We build and run the agentic loop end-to-end.

### What the agent does

```
User gives: starting SMILES + optimization goal
       │
       ▼
 ┌─────────────────────────────────────────────────┐
 │  Round N                                        │
 │  1. Agent reasons about current ADMET profile   │
 │  2. Identifies the largest gap from the goal    │
 │  3. Proposes a structural modification          │
 │  4. validate_smiles → is the chemistry valid?   │
 │  5. analyze_molecule → new ADMET scores         │
 │  6. compare_candidates → am I improving?        │
 │  7. Decide: continue or stop                    │
 └──────────────────┬──────────────────────────────┘
                    │ repeat
                    ▼
       Best candidate + reasoning summary
```

### Demo scenario
- **Start:** Aspirin
- **Goal:** *Improve CNS penetration (BBB+, CNS MPO > 4) while keeping QED > 0.5 and zero structural alerts*

> **Why this scenario?** Aspirin is well-known (great for interviews), BBB− by default, and CNS MPO optimization requires real structural reasoning (reduce HBD, adjust lipophilicity, control MW).

---
## Setup

In [ ]:
import sys, os, json, time
import anthropic

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from agent_utils import tool_executor

# Set your API key (or export ANTHROPIC_API_KEY in your shell)
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
print('Anthropic client ready')

---
## 1. Tool Definitions

The agent has **3 tools** — these are its only actions. Everything else is reasoning.

In [ ]:
TOOLS = [
    {
        "name": "validate_smiles",
        "description": (
            "Validate a SMILES string using RDKit (local, instant). "
            "Always call this FIRST before analyze_molecule."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "smiles": {"type": "string", "description": "SMILES string to validate"}
            },
            "required": ["smiles"]
        }
    },
    {
        "name": "analyze_molecule",
        "description": (
            "Get full ADMET profile for a molecule: QED, Lipinski, BBB penetration, "
            "CNS MPO score, solubility, GI absorption, CYP450, structural alerts, "
            "triage decision (Progress/Optimize/Kill), and API optimization suggestions. "
            "Note: first call may take ~60s (server cold start)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "smiles": {"type": "string", "description": "Valid SMILES string"}
            },
            "required": ["smiles"]
        }
    },
    {
        "name": "compare_candidates",
        "description": "Compare multiple molecules side by side to rank candidates.",
        "input_schema": {
            "type": "object",
            "properties": {
                "smiles_list": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of SMILES to compare"
                },
                "labels": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Labels for each molecule"
                }
            },
            "required": ["smiles_list"]
        }
    }
]

---
## 2. System Prompt — The Agent's Identity

The system prompt defines:
- **Role:** medicinal chemist AI
- **Strategy:** how to reason about ADMET properties
- **Chemistry knowledge:** key structure–property relationships
- **Workflow:** validate → analyze → compare → iterate

In [ ]:
SYSTEM_PROMPT = """
You are an expert medicinal chemistry AI specializing in lead optimization.
Your job is to iteratively improve a drug candidate's ADMET profile toward a user-specified goal.

## Your Tools
1. validate_smiles(smiles)          — ALWAYS call this before analyze_molecule
2. analyze_molecule(smiles)         — Full ADMET profile from the Drug Discovery Triage API
3. compare_candidates(smiles_list)  — Rank multiple candidates

## Optimization Workflow (follow strictly)
For each round:
  Step 1: Identify the biggest gap between current scores and the goal
  Step 2: Reason about which structural change addresses this gap
  Step 3: Propose a modified SMILES with a specific chemical justification
  Step 4: validate_smiles → if invalid, try a simpler modification
  Step 5: analyze_molecule → get new ADMET scores
  Step 6: compare_candidates every 2 rounds to track overall progress
  Step 7: Decide whether to continue or stop

## Key Structure–Property Relationships
| Goal                       | Strategy |
|----------------------------|---------|
| Improve BBB / CNS MPO      | Reduce HBD (replace -OH with -F or -OMe), reduce MW (<450), reduce PSA (<90 Å²), increase lipophilicity |
| Improve solubility         | Add -OH, -NH2, ionisable groups; reduce aromaticity |
| Improve QED                | Optimize MW (200-500), LogP (0-5), reduce rotatable bonds |
| Remove PAINS alerts        | Avoid quinones, catechols, Michael acceptors, anilines |
| Bioisosteric replacements  | -COOH ↔ tetrazole, -OH ↔ -F, phenyl ↔ pyridine, ester ↔ amide |

## Stopping Rules
- Stop when ALL goal criteria are met
- Stop if no improvement in 2 consecutive rounds
- Always end with: best candidate SMILES, a summary table, and your chemical reasoning

## Style
- Think step by step, like a senior medicinal chemist
- Make ONE structural change per round for clear SAR (structure-activity relationship) understanding
- Pay attention to the API's own decision_rationale and api_suggestions — these are validated hints
"""

print('System prompt defined.')
print(f'Length: {len(SYSTEM_PROMPT)} characters')

---
## 3. The Agent Loop

This is the core of the agentic system. The loop:
1. Sends messages to Claude
2. Intercepts `tool_use` blocks
3. Executes the tool (hits the ADMET API or RDKit)
4. Feeds results back as `tool_result`
5. Repeats until Claude outputs `end_turn`

In [ ]:
def run_optimization_agent(
    starting_smiles: str,
    goal: str,
    max_tool_calls: int = 20,
    model: str = "claude-opus-4-6",
) -> tuple[list, list]:
    """
    Run the lead optimization agent.

    Parameters
    ----------
    starting_smiles : SMILES of the molecule to optimize
    goal            : Natural-language optimization target
    max_tool_calls  : Safety limit on total tool invocations
    model           : Claude model to use

    Returns
    -------
    messages   : Full conversation history (for inspection)
    candidates : List of analyzed molecules with their scores
    """
    messages = []
    candidates = []   # track every molecule analyzed
    tool_call_count = 0

    # ── Initial user message ──────────────────────────────────────────────────
    messages.append({
        "role": "user",
        "content": f"""
Starting molecule SMILES: {starting_smiles}

Optimization goal: {goal}

Please:
1. First analyze the starting molecule to establish the baseline ADMET profile
2. Identify exactly which properties fall short of the goal
3. Propose a structural modification with explicit chemical reasoning
4. Validate → Analyze → Compare across rounds
5. After completing your optimization, present the best candidate and a
   summary of what changed chemically and why.
"""
    })

    print('=' * 65)
    print(f'  Starting molecule : {starting_smiles}')
    print(f'  Goal              : {goal}')
    print(f'  Model             : {model}')
    print('=' * 65)

    # ── Agentic loop ──────────────────────────────────────────────────────────
    while True:
        response = client.messages.create(
            model=model,
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )

        # Append assistant turn to history
        messages.append({"role": "assistant", "content": response.content})

        # ── Print any reasoning text from the agent ───────────────────────────
        for block in response.content:
            if hasattr(block, 'text') and block.text.strip():
                print(f'\n🤖 Agent:\n{block.text}')

        # ── Stop if agent is done ─────────────────────────────────────────────
        if response.stop_reason == 'end_turn':
            print('\n✅ Agent completed optimization.')
            break

        # ── Handle tool calls ─────────────────────────────────────────────────
        if response.stop_reason == 'tool_use':
            tool_results = []

            for block in response.content:
                if block.type != 'tool_use':
                    continue

                tool_call_count += 1
                print(f'\n🔧 Tool [{tool_call_count}]: {block.name}')
                print(f'   Input: {json.dumps(block.input)}')

                t0 = time.time()
                result = tool_executor(block.name, block.input)
                elapsed = time.time() - t0

                print(f'   Time : {elapsed:.1f}s')
                print(f'   Result (preview): {json.dumps(result)[:300]}')

                # Track every molecule that was analyzed
                if block.name == 'analyze_molecule' and 'error' not in result:
                    candidates.append({
                        'round': tool_call_count,
                        'input_smiles': block.input['smiles'],
                        **result,
                    })

                tool_results.append({
                    'type': 'tool_result',
                    'tool_use_id': block.id,
                    'content': json.dumps(result),
                })

            # Feed tool results back into the conversation
            messages.append({'role': 'user', 'content': tool_results})

        # ── Safety brake ─────────────────────────────────────────────────────
        if tool_call_count >= max_tool_calls:
            print(f'\n⚠️  Safety limit reached ({max_tool_calls} tool calls). Stopping.')
            break

    return messages, candidates

---
## 4. Run the Agent!

**Scenario:** Optimize **Atenolol** for CNS penetration.

Why Atenolol is the ideal starting molecule for this demo:
- Well-known cardioselective beta-blocker (instantly recognizable)
- **Deliberately designed to NOT cross the BBB** — unlike propranolol (older, CNS-active)
- High polarity: 4 H-bond donors, PSA ~84 Å², MW 266 → BBB−, low CNS MPO
- Agent must reduce polarity (replace -OH/-NH2 with bioisosteres) while keeping the pharmacophore
- Classic medicinal chemistry challenge with a clear before/after story

> ⏱️ Each `analyze_molecule` call takes ~5-10 s after server warm-up. Full run ~3-5 minutes.

In [ ]:
# ── Choose your starting molecule ────────────────────────────────────────
#
# Atenolol vs Propranolol — same aryloxypropanolamine scaffold, different CNS profile:
#
# Property         Atenolol   Propranolol   (target)
# ─────────────────────────────────────────────────
# cLogP            0.45       2.58          → need higher lipophilicity
# TPSA (Å²)       84.6       41.5          → need to reduce polarity
# H-bond donors    3          2             → remove one HBD
# BBB probability  0.60       0.90          → our main target
# CNS MPO score    4.55       5.45          → push above 5.0
# QED              0.638      0.838         → aim for > 0.75
#
# Task for the agent: nudge Atenolol toward a Propranolol-like CNS profile
# by reducing the polar amide group and increasing lipophilicity.

STARTING_SMILES = 'CC(C)NCC(O)COc1ccc(CC(N)=O)cc1'   # Atenolol

GOAL = """
Transform this Atenolol scaffold toward a CNS-penetrating profile.
Reference target: Propranolol (BBB prob 0.90, CNS MPO 5.45, QED 0.838).

Specific targets to hit:
  - BBB probability > 0.80  (currently 0.60  — borderline)
  - CNS MPO score   > 5.0   (currently 4.55  — needs ~0.5 point gain)
  - QED             > 0.75  (currently 0.638 — needs improvement)
  - Rotatable bonds ≤ 6     (currently 8     — too flexible)
  - Zero structural alerts
  - MW < 400 Da

Key chemistry hint: the para-amide (-CC(N)=O) group is a major polarity liability
(high HBD, high PSA). Consider reducing or replacing it.
Preserve the aryloxypropanolamine pharmacophore (required for beta-blocker activity).
"""

messages, candidates = run_optimization_agent(
    starting_smiles=STARTING_SMILES,
    goal=GOAL,
    max_tool_calls=20,
)

---
## 5. Results — Optimization Trajectory

Every molecule the agent analyzed, in order.

In [ ]:
import pandas as pd

if not candidates:
    print('No candidates collected — check for API errors above.')
else:
    df = pd.DataFrame(candidates)

    # Select the most relevant columns for the optimization story
    cols = [
        'round', 'input_smiles', 'molecular_weight', 'clogp',
        'qed_score', 'bbb_penetrates', 'bbb_probability',
        'cns_mpo_score', 'cns_class', 'num_alerts', 'decision'
    ]
    cols = [c for c in cols if c in df.columns]

    display_df = df[cols].copy()

    # Colour-code the decision column for readability
    def style_decision(val):
        colors = {'Progress': 'background-color: #d4edda',
                  'Optimize': 'background-color: #fff3cd',
                  'Kill':     'background-color: #f8d7da'}
        return colors.get(val, '')

    styled = display_df.style.applymap(style_decision, subset=['decision'] if 'decision' in cols else [])
    display(styled)

    # Quick summary
    if len(df) > 1:
        start = df.iloc[0]
        best  = df.loc[df['cns_mpo_score'].idxmax()] if 'cns_mpo_score' in df.columns else df.iloc[-1]
        print(f'\nStarting CNS MPO : {start.get("cns_mpo_score", "N/A")}')
        print(f'Best CNS MPO     : {best.get("cns_mpo_score", "N/A")} (round {best.get("round", "?")})')
        print(f'BBB penetration  : {start.get("bbb_penetrates", "?")} → {best.get("bbb_penetrates", "?")}')
        print(f'QED              : {start.get("qed_score", "N/A")} → {best.get("qed_score", "N/A")}')

In [ ]:
# Save candidates to JSON so notebook 03 can load them for visualization
import json
output_path = '../candidates.json'
with open(output_path, 'w') as f:
    json.dump(candidates, f, indent=2, default=str)

print(f'Saved {len(candidates)} candidates to {output_path}')
print('Ready for notebook 03 (visualization).')

---
## What just happened — the agentic loop explained

```
User message ──► Claude reasons ──► tool_use block
                                          │
                                          ▼
                                   tool_executor()
                                    ├── validate_smiles → RDKit (local)
                                    ├── analyze_molecule → ADMET API
                                    └── compare_candidates → ADMET API × N
                                          │
                             tool_result ◄┘
                                 added to messages
                                          │
                                 Claude reasons again
                                 (repeat until end_turn)
```

**Key design choices:**
- `validate_smiles` first → prevents wasting API calls on bad chemistry
- One structural change per round → clear SAR understanding
- `compare_candidates` every 2 rounds → agent can backtrack if a round was worse
- Safety brake at `max_tool_calls` → no runaway loops

➡️ **Next:** `03_visualization.ipynb` — visualize the optimization trajectory